In [ ]:
!pip install -q \
    datasets \
    openai \
    pandas \
    numpy \
    tqdm \
    transformers \
    sentence-transformers \
    torch \
    faiss-cpu \
    scipy \
    statsmodels \
    matplotlib

In [ ]:
import os
import re
import json
import time
import random
import hashlib
import warnings
from pathlib import Path
from getpass import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

import faiss

from scipy.stats import bootstrap
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

from openai import AzureOpenAI

warnings.filterwarnings("ignore")

In [ ]:

from google.colab import files

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR = Path("/content/ground_truth")
RESULTS_DIR = BASE_DIR / "results"
CACHE_DIR = BASE_DIR / "cache"
FIGURES_DIR = BASE_DIR / "figures"

for directory in [BASE_DIR, RESULTS_DIR, CACHE_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Base directory:", BASE_DIR)

In [ ]:
AZURE_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT") or os.environ.get("ENDPOINT_URL")
AZURE_API_KEY = os.environ.get("AZURE_OPENAI_API_KEY")
if not AZURE_ENDPOINT or not AZURE_API_KEY:
    raise EnvironmentError("Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY before running.")
client = AzureOpenAI(
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION"),
)
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT")

MAX_COMPLETION_TOKENS = 15000

print("Azure client initialized.")

In [ ]:
# ------------------------------------------------------------
# Experimental configuration
# ------------------------------------------------------------

SNLI_SPLIT = "test"

# First use 100 or 200 for a pilot.
# Later increase to 1000, or set to None for every valid premise group.
N_TEST_GROUPS = None

# Persona-DB analogue:
# Number of train hypotheses used to build the offline prototype index.
PROTOTYPE_TRAIN_SIZE = 20000

DENSE_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

METHOD_NLI_MODEL_NAME = (
    "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
)

# In our method, NLI keeps the top 2 and GPT makes the final decision.
NLI_GPT_SHORTLIST_SIZE = 2

METHODS = [
    "SPR_semantic_retrieval",
    "partner_matters_semantic_fusion",
    "coo_style_usefulness_ranking",
    "personadb_style_prototype_retrieval",
    "ours_nli_gpt",
]

LABEL_NAMES = {
    0: "entailment",
    1: "neutral",
    2: "contradiction",
}

RESULTS_PATH = RESULTS_DIR / (
    f"snli_results_{SNLI_SPLIT}_{N_TEST_GROUPS or 'all'}.csv"
)

print("Methods:", len(METHODS))
print("Results path:", RESULTS_PATH)

In [ ]:
snli = load_dataset("stanfordnlp/snli")

print(snli)
print(snli["test"].features)

In [ ]:
def build_snli_candidate_groups(
    dataset_split,
    n_groups=None,
    seed=42,
):
    """
    Builds one retrieval task per premise.

    Each task contains exactly:
    - one entailment hypothesis,
    - one neutral hypothesis,
    - one contradiction hypothesis.

    Candidate IDs are assigned AFTER shuffling so that
    the ID contains no information about the gold label.
    """

    df = dataset_split.to_pandas().copy()

    # --------------------------------------------------------
    # Keep only valid SNLI labels
    # --------------------------------------------------------

    df["label"] = pd.to_numeric(
        df["label"],
        errors="coerce"
    )

    df = df[
        df["label"].isin([0, 1, 2])
    ].copy()

    # --------------------------------------------------------
    # Clean text
    # --------------------------------------------------------

    df["premise"] = (
        df["premise"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df["hypothesis"] = (
        df["hypothesis"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    df = df[
        df["premise"].ne("") &
        df["hypothesis"].ne("")
    ].copy()

    # Keep original row for debugging / traceability.
    df["_source_row"] = np.arange(len(df))

    rng = np.random.default_rng(seed)

    groups = []

    # ========================================================
    # Group all SNLI rows having exactly the same premise
    # ========================================================

    for premise, premise_df in df.groupby(
        "premise",
        sort=False
    ):

        available_labels = set(
            premise_df["label"]
            .astype(int)
            .tolist()
        )

        # We require at least one example of:
        # entailment, neutral and contradiction.
        if not {0, 1, 2}.issubset(available_labels):
            continue

        # ----------------------------------------------------
        # First construct candidates WITHOUT public IDs.
        # ----------------------------------------------------

        selected_candidates = []

        for label in [0, 1, 2]:

            label_df = premise_df[
                premise_df["label"]
                .astype(int)
                .eq(label)
            ]

            selected_index = int(
                rng.integers(
                    low=0,
                    high=len(label_df)
                )
            )

            selected_row = label_df.iloc[
                selected_index
            ]

            selected_candidates.append({
                "hypothesis": str(
                    selected_row["hypothesis"]
                ),
                "gold_label": int(label),
                "gold_label_name": (
                    LABEL_NAMES[int(label)]
                ),
                "source_row": int(
                    selected_row["_source_row"]
                ),
            })

        # ----------------------------------------------------
        # IMPORTANT:
        # Randomize the candidates BEFORE assigning IDs.
        # ----------------------------------------------------

        rng.shuffle(selected_candidates)

        # ----------------------------------------------------
        # Only now give neutral / opaque IDs.
        #
        # Therefore:
        # option_A is NOT systematically entailment,
        # neutral, or contradiction.
        # ----------------------------------------------------

        neutral_ids = [
            "option_A",
            "option_B",
            "option_C",
        ]

        candidates = []

        for candidate_id, candidate in zip(
            neutral_ids,
            selected_candidates
        ):

            candidates.append({
                "candidate_id": candidate_id,
                "hypothesis": candidate["hypothesis"],
                "gold_label": candidate["gold_label"],
                "gold_label_name": (
                    candidate["gold_label_name"]
                ),
                "source_row": candidate["source_row"],
            })

        # ----------------------------------------------------
        # Stable premise ID
        # ----------------------------------------------------

        premise_hash = hashlib.sha1(
            premise.encode("utf-8")
        ).hexdigest()[:12]

        groups.append({
            "case_id": f"snli_{premise_hash}",
            "premise": premise,
            "candidates": candidates,
        })

    # --------------------------------------------------------
    # Randomize order of test cases too
    # --------------------------------------------------------

    rng.shuffle(groups)

    if n_groups is not None:
        groups = groups[:n_groups]

    print("Valid premise groups:", len(groups))

    return groups

In [ ]:
TEST_GROUPS = build_snli_candidate_groups(
    dataset_split=snli[SNLI_SPLIT],
    n_groups=N_TEST_GROUPS,
    seed=SEED,
)

for group in TEST_GROUPS[:10]:

    print("\n================================")
    print("Case:", group["case_id"])
    print("Premise:", group["premise"])

    for candidate in group["candidates"]:
        print(
            candidate["candidate_id"],
            "->",
            candidate["gold_label_name"],
            "|",
            candidate["hypothesis"][:100]
        )

In [ ]:
id_label_rows = []

for group in TEST_GROUPS:
    for candidate in group["candidates"]:
        id_label_rows.append({
            "candidate_id": candidate["candidate_id"],
            "gold_label_name": candidate["gold_label_name"],
        })

id_label_df = pd.DataFrame(id_label_rows)

id_label_table = pd.crosstab(
    id_label_df["candidate_id"],
    id_label_df["gold_label_name"]
)

display(id_label_table)

In [ ]:
def validate_candidate_groups(groups):
    for group in groups:
        labels = sorted(
            candidate["gold_label"]
            for candidate in group["candidates"]
        )

        if labels != [0, 1, 2]:
            raise ValueError(
                f"Invalid labels in {group['case_id']}: {labels}"
            )

        ids = [
            candidate["candidate_id"]
            for candidate in group["candidates"]
        ]

        if len(ids) != len(set(ids)):
            raise ValueError(
                f"Duplicate candidate IDs in {group['case_id']}"
            )

    print(f"Validated {len(groups)} candidate groups.")


validate_candidate_groups(TEST_GROUPS)

In [ ]:
dense_model = SentenceTransformer(
    DENSE_MODEL_NAME,
    device=DEVICE,
)

print("Dense model loaded:", DENSE_MODEL_NAME)

In [ ]:
method_tokenizer = AutoTokenizer.from_pretrained(
    METHOD_NLI_MODEL_NAME
)

method_nli_model = (
    AutoModelForSequenceClassification
    .from_pretrained(METHOD_NLI_MODEL_NAME)
    .to(DEVICE)
)

method_nli_model.eval()

print("NLI model loaded:", METHOD_NLI_MODEL_NAME)
print("id2label:", method_nli_model.config.id2label)

In [ ]:
def find_label_index(model, target_label):
    for index, label in model.config.id2label.items():
        if target_label.lower() in str(label).lower():
            return int(index)

    raise ValueError(
        f"Could not find {target_label} in "
        f"{model.config.id2label}"
    )


CONTRADICTION_INDEX = find_label_index(
    method_nli_model,
    "contradiction",
)

print("Contradiction index:", CONTRADICTION_INDEX)

In [ ]:
@torch.inference_mode()
def nli_contradiction_scores(
    premise,
    hypotheses,
    batch_size=16,
    max_length=512,
):
    all_scores = []

    for start in range(0, len(hypotheses), batch_size):
        batch_hypotheses = hypotheses[
            start:start + batch_size
        ]

        premises = [premise] * len(batch_hypotheses)

        encoded = method_tokenizer(
            premises,
            batch_hypotheses,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encoded = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        logits = method_nli_model(**encoded).logits
        probabilities = torch.softmax(logits, dim=-1)

        contradiction_probs = probabilities[
            :, CONTRADICTION_INDEX
        ]

        all_scores.extend(
            contradiction_probs.detach().cpu().numpy().tolist()
        )

    return [float(score) for score in all_scores]

In [ ]:
def extract_json_object(text):
    text = (text or "").strip()

    if not text:
        raise ValueError("Empty GPT response.")

    # Remove Markdown fences.
    text = re.sub(
        r"^```(?:json)?\\s*|\\s*```$",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    ).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\\{.*\\}", text, flags=re.DOTALL)

        if not match:
            raise ValueError(
                f"No JSON object found in GPT response: {text[:500]}"
            )

        return json.loads(match.group(0))


def call_gpt_json(
    prompt,
    system_message,
    retries=3,
):
    last_error = None

    for attempt in range(1, retries + 1):
        try:
            response = client.chat.completions.create(
                model=AZURE_OPENAI_DEPLOYMENT,
                messages=[
                    {
                        "role": "system",
                        "content": system_message,
                    },
                    {
                        "role": "user",
                        "content": prompt,
                    },
                ],
                max_completion_tokens=MAX_COMPLETION_TOKENS,
                response_format={"type": "json_object"},
            )

            content = response.choices[0].message.content

            if not content:
                raise ValueError(
                    f"Empty GPT response. Full response: {response}"
                )

            return extract_json_object(content)

        except Exception as error:
            last_error = error
            print(
                f"GPT attempt {attempt}/{retries} failed:",
                repr(error),
            )

            if attempt < retries:
                time.sleep(2 ** attempt)

    raise RuntimeError(
        f"GPT failed after {retries} attempts: {repr(last_error)}"
    )

In [ ]:
def candidate_lookup(candidates):
    return {
        str(candidate["candidate_id"]): candidate
        for candidate in candidates
    }


def validate_ranking(ranking, candidates):
    valid_ids = {
        str(candidate["candidate_id"])
        for candidate in candidates
    }

    cleaned = []

    for candidate_id in ranking:
        candidate_id = str(candidate_id)

        if (
            candidate_id in valid_ids
            and candidate_id not in cleaned
        ):
            cleaned.append(candidate_id)

    # Append candidates omitted by a method.
    for candidate in candidates:
        candidate_id = str(candidate["candidate_id"])

        if candidate_id not in cleaned:
            cleaned.append(candidate_id)

    return cleaned


def rank_from_scores(candidates, scores, descending=True):
    if len(candidates) != len(scores):
        raise ValueError("Candidates and scores differ in length.")

    scored = list(zip(candidates, scores))

    scored.sort(
        key=lambda item: item[1],
        reverse=descending,
    )

    return [
        str(candidate["candidate_id"])
        for candidate, score in scored
    ]

In [ ]:
def method_SPR_semantic_retrieval(
    premise,
    candidates,
    case_seed=None,
):
    hypotheses = [
        candidate["hypothesis"]
        for candidate in candidates
    ]

    embeddings = dense_model.encode(
        [premise] + hypotheses,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    premise_embedding = embeddings[0]
    hypothesis_embeddings = embeddings[1:]

    cosine_scores = (
        hypothesis_embeddings @ premise_embedding
    )

    ranking = rank_from_scores(
        candidates,
        cosine_scores,
        descending=True,
    )

    return {
        "ranking": ranking,
        "scores": {
            str(candidate["candidate_id"]): float(score)
            for candidate, score in zip(
                candidates,
                cosine_scores,
            )
        },
        "details": (
            "Ranks candidates by cosine similarity "
            "between premise and hypothesis."
        ),
    }

In [ ]:
def method_partner_matters_semantic_fusion(
    premise,
    candidates,
    case_seed=None,
):
    hypotheses = [
        candidate["hypothesis"]
        for candidate in candidates
    ]

    embeddings = dense_model.encode(
        [premise] + hypotheses,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    premise_embedding = embeddings[0]
    hypothesis_embeddings = embeddings[1:]

    fused_candidate_embedding = (
        hypothesis_embeddings.mean(axis=0)
    )

    fused_candidate_embedding = (
        fused_candidate_embedding /
        (np.linalg.norm(fused_candidate_embedding) + 1e-12)
    )

    context_similarity = (
        hypothesis_embeddings @ premise_embedding
    )

    fusion_similarity = (
        hypothesis_embeddings @ fused_candidate_embedding
    )

    # Semantic relevance receives more weight.
    fused_scores = (
        0.70 * context_similarity +
        0.30 * fusion_similarity
    )

    ranking = rank_from_scores(
        candidates,
        fused_scores,
        descending=True,
    )

    return {
        "ranking": ranking,
        "scores": {
            str(candidate["candidate_id"]): float(score)
            for candidate, score in zip(
                candidates,
                fused_scores,
            )
        },
        "details": (
            "SNLI-adapted semantic fusion of premise relevance "
            "and candidate-set representation."
        ),
    }

In [ ]:
def method_coo_style_usefulness_ranking(
    premise,
    candidates,
    case_seed=None,
):
    compact_candidates = [
        {
            "id": str(candidate["candidate_id"]),
            "text": candidate["hypothesis"],
        }
        for candidate in candidates
    ]

    prompt = f"""
Premise:
{premise}

Candidate statements:
{json.dumps(compact_candidates, ensure_ascii=False)}

Rank the candidate statements by how useful, clear, and relevant they
would be as evidence for understanding or responding to the premise.

Important:
- Rank by general usefulness, clarity, semantic relevance, and informativeness.
- Do not explicitly optimize for contradiction.
- Do not use or infer any hidden gold labels.
- Return every candidate exactly once.

Return valid JSON only:
{{
  "ranked_ids": ["candidate_id_1", "candidate_id_2", "candidate_id_3"],
  "reason": "brief explanation"
}}
"""

    data = call_gpt_json(
        prompt=prompt,
        system_message=(
            "You are a careful usefulness-ranking system. "
            "Return valid JSON only."
        ),
    )

    ranking = validate_ranking(
        data.get("ranked_ids", []),
        candidates,
    )

    return {
        "ranking": ranking,
        "scores": {},
        "details": str(data.get("reason", ""))[:500],
    }

In [ ]:
PERSONADB_INDEX_PATH = (
    CACHE_DIR /
    f"personadb_train_hypotheses_{PROTOTYPE_TRAIN_SIZE}.faiss"
)

PERSONADB_META_PATH = (
    CACHE_DIR /
    f"personadb_train_hypotheses_{PROTOTYPE_TRAIN_SIZE}.csv"
)


def build_or_load_personadb_prototype_index(
    force_rebuild=False,
):
    if (
        PERSONADB_INDEX_PATH.exists()
        and PERSONADB_META_PATH.exists()
        and not force_rebuild
    ):
        print("Loading cached Persona-DB-style index.")

        index = faiss.read_index(
            str(PERSONADB_INDEX_PATH)
        )

        metadata = pd.read_csv(
            PERSONADB_META_PATH,
            encoding="utf-8-sig",
        )

        return metadata, index

    train_df = snli["train"].to_pandas().copy()

    train_df["label"] = pd.to_numeric(
        train_df["label"],
        errors="coerce",
    )

    train_df = train_df[
        train_df["label"].isin([0, 1, 2])
    ].copy()

    train_df["hypothesis"] = (
        train_df["hypothesis"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    train_df = train_df[
        train_df["hypothesis"].ne("")
    ].copy()

    if len(train_df) > PROTOTYPE_TRAIN_SIZE:
        train_df = train_df.sample(
            n=PROTOTYPE_TRAIN_SIZE,
            random_state=SEED,
        )

    train_df = train_df.reset_index(drop=True)

    print(
        "Encoding Persona-DB-style offline database:",
        len(train_df),
    )

    embeddings = dense_model.encode(
        train_df["hypothesis"].tolist(),
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    faiss.write_index(
        index,
        str(PERSONADB_INDEX_PATH),
    )

    train_df[
        ["premise", "hypothesis", "label"]
    ].to_csv(
        PERSONADB_META_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    print("Saved Persona-DB-style prototype index.")

    return train_df, index


personadb_meta, personadb_index = (
    build_or_load_personadb_prototype_index(
        force_rebuild=False
    )
)

In [ ]:
def method_personadb_style_prototype_retrieval(
    premise,
    candidates,
    case_seed=None,
    nearest_k=8,
):
    hypotheses = [
        candidate["hypothesis"]
        for candidate in candidates
    ]

    embeddings = dense_model.encode(
        hypotheses,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    similarities, indices = personadb_index.search(
        embeddings,
        nearest_k,
    )

    # Mean similarity to the nearest refined database items.
    prototype_scores = similarities.mean(axis=1)

    ranking = rank_from_scores(
        candidates,
        prototype_scores,
        descending=True,
    )

    return {
        "ranking": ranking,
        "scores": {
            str(candidate["candidate_id"]): float(score)
            for candidate, score in zip(
                candidates,
                prototype_scores,
            )
        },
        "details": (
            "SNLI-adapted offline prototype retrieval. "
            "No contradiction labels are used during ranking."
        ),
    }

In [ ]:
def method_ours_nli_gpt(
    premise,
    candidates,
    case_seed=None,
    shortlist_size=NLI_GPT_SHORTLIST_SIZE,
):
    hypotheses = [
        candidate["hypothesis"]
        for candidate in candidates
    ]

    nli_scores = nli_contradiction_scores(
        premise=premise,
        hypotheses=hypotheses,
    )

    nli_ranking = rank_from_scores(
        candidates,
        nli_scores,
        descending=True,
    )

    lookup = candidate_lookup(candidates)

    shortlist_ids = nli_ranking[:shortlist_size]

    shortlist = []

    score_lookup = {
        str(candidate["candidate_id"]): float(score)
        for candidate, score in zip(
            candidates,
            nli_scores,
        )
    }

    for candidate_id in shortlist_ids:
        candidate = lookup[candidate_id]

        shortlist.append({
            "id": candidate_id,
            "text": candidate["hypothesis"],
            "nli_contradiction_score": round(
                score_lookup[candidate_id],
                6,
            ),
        })

    prompt = f"""
Premise:
{premise}

NLI-shortlisted hypotheses:
{json.dumps(shortlist, ensure_ascii=False)}

Select the single hypothesis that most directly contradicts the premise.

Important:
- Select one of the provided candidate IDs.
- Do not generate or rewrite a hypothesis.
- Prefer logical contradiction over mere difference or uncertainty.
- The NLI score is supporting information, not a gold label.

Return valid JSON only:
{{
  "best_id": "candidate_id",
  "reason": "brief explanation"
}}
"""

    data = call_gpt_json(
        prompt=prompt,
        system_message=(
            "You are the final contradiction reranker. "
            "Select an existing candidate and return valid JSON only."
        ),
    )

    selected_id = str(
        data.get("best_id", shortlist_ids[0])
    )

    if selected_id not in shortlist_ids:
        selected_id = shortlist_ids[0]

    # Selected candidate first.
    # Remaining shortlist candidates retain NLI order.
    # Candidates excluded by NLI are appended last.
    final_ranking = [selected_id]

    for candidate_id in nli_ranking:
        if candidate_id not in final_ranking:
            final_ranking.append(candidate_id)

    return {
        "ranking": final_ranking,
        "scores": score_lookup,
        "details": {
            "nli_ranking": nli_ranking,
            "shortlist_ids": shortlist_ids,
            "gpt_selected_id": selected_id,
            "gpt_reason": str(
                data.get("reason", "")
            )[:500],
        },
    }

In [ ]:
METHOD_FUNCTIONS = {
    "SPR_semantic_retrieval": (
        method_SPR_semantic_retrieval
    ),
    "partner_matters_semantic_fusion": (
        method_partner_matters_semantic_fusion
    ),
    "coo_style_usefulness_ranking": (
        method_coo_style_usefulness_ranking
    ),
    "personadb_style_prototype_retrieval": (
        method_personadb_style_prototype_retrieval
    ),
    "ours_nli_gpt": method_ours_nli_gpt,
}

assert list(METHOD_FUNCTIONS.keys()) == METHODS

print("Registered methods:")
for method_name in METHODS:
    print("-", method_name)

In [ ]:
def evaluate_ranking(candidates, ranking):
    lookup = candidate_lookup(candidates)

    ranking = validate_ranking(
        ranking,
        candidates,
    )

    ranked_labels = [
        int(lookup[candidate_id]["gold_label"])
        for candidate_id in ranking
    ]

    contradiction_positions = [
        position
        for position, label in enumerate(
            ranked_labels,
            start=1,
        )
        if label == 2
    ]

    if not contradiction_positions:
        contradiction_rank = np.nan
        reciprocal_rank = 0.0
    else:
        contradiction_rank = contradiction_positions[0]
        reciprocal_rank = 1.0 / contradiction_rank

    top1_label = ranked_labels[0]

    return {
        "selected_candidate_id": ranking[0],
        "selected_gold_label": top1_label,
        "selected_gold_label_name": LABEL_NAMES[top1_label],
        "top1_contradiction_correct": int(top1_label == 2),
        "contradiction_rank": contradiction_rank,
        "reciprocal_rank": reciprocal_rank,
        "recall_at_2": int(
            any(label == 2 for label in ranked_labels[:2])
        ),
        "full_ranking": json.dumps(ranking),
        "ranked_gold_labels": json.dumps(ranked_labels),
    }

In [ ]:
def stable_case_seed(case_id):
    return int(
        hashlib.sha1(
            case_id.encode("utf-8")
        ).hexdigest()[:8],
        16,
    )


def run_single_case(group):
    rows = []

    case_id = group["case_id"]
    premise = group["premise"]
    candidates = group["candidates"]

    case_seed = stable_case_seed(case_id)

    for method_name, method_function in METHOD_FUNCTIONS.items():
        print(
            f"{case_id} | {method_name}"
        )

        start_time = time.perf_counter()

        try:
            output = method_function(
                premise=premise,
                candidates=candidates,
                case_seed=case_seed,
            )

            ranking = output["ranking"]
            method_scores = output.get("scores", {})
            method_details = output.get("details", "")
            method_error = ""

        except Exception as error:
            print("METHOD ERROR:", repr(error))

            ranking = [
                str(candidate["candidate_id"])
                for candidate in candidates
            ]

            method_scores = {}
            method_details = ""
            method_error = repr(error)

        runtime_seconds = (
            time.perf_counter() - start_time
        )

        metrics = evaluate_ranking(
            candidates=candidates,
            ranking=ranking,
        )

        selected_candidate = candidate_lookup(candidates)[
            metrics["selected_candidate_id"]
        ]

        rows.append({
            "case_id": case_id,
            "premise": premise,
            "method": method_name,
            "runtime_seconds": runtime_seconds,
            "selected_hypothesis": (
                selected_candidate["hypothesis"]
            ),
            "method_scores": json.dumps(
                method_scores,
                ensure_ascii=False,
            ),
            "method_details": json.dumps(
                method_details,
                ensure_ascii=False,
            ) if isinstance(
                method_details,
                (dict, list),
            ) else str(method_details),
            "method_error": method_error,
            **metrics,
        })

    return rows

In [ ]:
if RESULTS_PATH.exists():
    results_df = pd.read_csv(
        RESULTS_PATH,
        encoding="utf-8-sig",
    )

    completed_keys = set(
        zip(
            results_df["case_id"].astype(str),
            results_df["method"].astype(str),
        )
    )

    all_rows = results_df.to_dict("records")

    print(
        "Resuming existing results:",
        len(results_df),
        "rows"
    )
else:
    completed_keys = set()
    all_rows = []

for group_index, group in enumerate(
    tqdm(TEST_GROUPS, desc="SNLI cases"),
    start=1,
):
    case_id = str(group["case_id"])

    missing_methods = [
        method_name
        for method_name in METHODS
        if (case_id, method_name) not in completed_keys
    ]

    if not missing_methods:
        continue

    case_rows = run_single_case(group)

    for row in case_rows:
        key = (
            str(row["case_id"]),
            str(row["method"]),
        )

        if key not in completed_keys:
            all_rows.append(row)
            completed_keys.add(key)

    results_df = pd.DataFrame(all_rows)

    results_df.to_csv(
        RESULTS_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    if group_index % 10 == 0:
        print(
            f"Saved after {group_index} groups:",
            RESULTS_PATH,
        )

results_df = pd.DataFrame(all_rows)

results_df.to_csv(
    RESULTS_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Finished.")
print("Saved:", RESULTS_PATH)
print("Rows:", len(results_df))

display(results_df.head())

In [ ]:
expected_rows = len(TEST_GROUPS) * len(METHODS)

print("Expected rows:", expected_rows)
print("Actual rows:", len(results_df))

duplicates = results_df.duplicated(
    subset=["case_id", "method"]
).sum()

print("Duplicate case-method rows:", duplicates)
print(
    "Method errors:",
    results_df["method_error"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

assert duplicates == 0, "Duplicate method results found."
assert len(results_df) == expected_rows, (
    "The run is incomplete."
)

In [ ]:
summary_df = (
    results_df
    .groupby("method", as_index=False)
    .agg(
        n_cases=("case_id", "nunique"),
        top1_contradiction_accuracy=(
            "top1_contradiction_correct",
            "mean",
        ),
        mean_reciprocal_rank=(
            "reciprocal_rank",
            "mean",
        ),
        recall_at_2=(
            "recall_at_2",
            "mean",
        ),
        mean_contradiction_rank=(
            "contradiction_rank",
            "mean",
        ),
        mean_runtime_seconds=(
            "runtime_seconds",
            "mean",
        ),
        error_count=(
            "method_error",
            lambda values: (
                values.fillna("")
                .astype(str)
                .str.strip()
                .ne("")
                .sum()
            ),
        ),
    )
)

summary_df = summary_df.sort_values(
    "top1_contradiction_accuracy",
    ascending=False,
).reset_index(drop=True)

SUMMARY_PATH = RESULTS_DIR / "snli_summary_results.csv"

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(summary_df.round(4))

print("Saved:", SUMMARY_PATH)

In [ ]:
selected_label_distribution = (
    results_df
    .groupby(
        ["method", "selected_gold_label_name"]
    )
    .size()
    .unstack(fill_value=0)
)

selected_label_distribution = (
    selected_label_distribution.div(
        selected_label_distribution.sum(axis=1),
        axis=0,
    )
)

LABEL_DISTRIBUTION_PATH = (
    RESULTS_DIR /
    "snli_selected_label_distribution.csv"
)

selected_label_distribution.to_csv(
    LABEL_DISTRIBUTION_PATH,
    encoding="utf-8-sig",
)

display(selected_label_distribution.round(4))

In [ ]:
def bootstrap_mean_ci(
    values,
    confidence_level=0.95,
    n_resamples=5000,
):
    values = np.asarray(values, dtype=float)

    if len(values) < 2:
        return np.nan, np.nan

    result = bootstrap(
        data=(values,),
        statistic=np.mean,
        confidence_level=confidence_level,
        n_resamples=n_resamples,
        random_state=SEED,
        method="percentile",
    )

    return (
        float(result.confidence_interval.low),
        float(result.confidence_interval.high),
    )


ci_rows = []

for method_name, method_df in results_df.groupby("method"):
    accuracy_values = (
        method_df["top1_contradiction_correct"]
        .astype(float)
        .to_numpy()
    )

    ci_low, ci_high = bootstrap_mean_ci(
        accuracy_values
    )

    ci_rows.append({
        "method": method_name,
        "top1_accuracy": accuracy_values.mean(),
        "ci_95_low": ci_low,
        "ci_95_high": ci_high,
    })

accuracy_ci_df = (
    pd.DataFrame(ci_rows)
    .sort_values("top1_accuracy", ascending=False)
)

ACCURACY_CI_PATH = (
    RESULTS_DIR /
    "snli_top1_accuracy_bootstrap_ci.csv"
)

accuracy_ci_df.to_csv(
    ACCURACY_CI_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(accuracy_ci_df.round(4))

In [ ]:
ours_df = (
    results_df[
        results_df["method"].eq("ours_nli_gpt")
    ][
        ["case_id", "top1_contradiction_correct"]
    ]
    .rename(
        columns={
            "top1_contradiction_correct": "ours_correct"
        }
    )
)

mcnemar_rows = []

for method_name in METHODS:
    if method_name == "ours_nli_gpt":
        continue

    comparison_df = (
        results_df[
            results_df["method"].eq(method_name)
        ][
            ["case_id", "top1_contradiction_correct"]
        ]
        .rename(
            columns={
                "top1_contradiction_correct": "baseline_correct"
            }
        )
        .merge(
            ours_df,
            on="case_id",
            how="inner",
        )
    )

    both_wrong = int(
        (
            comparison_df["baseline_correct"].eq(0) &
            comparison_df["ours_correct"].eq(0)
        ).sum()
    )

    baseline_only = int(
        (
            comparison_df["baseline_correct"].eq(1) &
            comparison_df["ours_correct"].eq(0)
        ).sum()
    )

    ours_only = int(
        (
            comparison_df["baseline_correct"].eq(0) &
            comparison_df["ours_correct"].eq(1)
        ).sum()
    )

    both_correct = int(
        (
            comparison_df["baseline_correct"].eq(1) &
            comparison_df["ours_correct"].eq(1)
        ).sum()
    )

    contingency_table = [
        [both_wrong, ours_only],
        [baseline_only, both_correct],
    ]

    test_result = mcnemar(
        contingency_table,
        exact=True,
    )

    mcnemar_rows.append({
        "baseline": method_name,
        "n_cases": len(comparison_df),
        "both_wrong": both_wrong,
        "baseline_only_correct": baseline_only,
        "ours_only_correct": ours_only,
        "both_correct": both_correct,
        "raw_p_value": float(test_result.pvalue),
    })

mcnemar_df = pd.DataFrame(mcnemar_rows)

if not mcnemar_df.empty:
    reject, holm_p, _, _ = multipletests(
        mcnemar_df["raw_p_value"],
        method="holm",
    )

    mcnemar_df["holm_p_value"] = holm_p
    mcnemar_df["significant_after_holm"] = reject

MCNEMAR_PATH = RESULTS_DIR / "snli_mcnemar_tests.csv"

mcnemar_df.to_csv(
    MCNEMAR_PATH,
    index=False,
    encoding="utf-8-sig",
)

display(mcnemar_df.round(4))

In [ ]:
plot_df = accuracy_ci_df.sort_values(
    "top1_accuracy",
    ascending=True,
).copy()

lower_errors = (
    plot_df["top1_accuracy"] -
    plot_df["ci_95_low"]
)

upper_errors = (
    plot_df["ci_95_high"] -
    plot_df["top1_accuracy"]
)

plt.figure(figsize=(11, 7))

plt.barh(
    plot_df["method"],
    plot_df["top1_accuracy"],
    xerr=np.vstack([
        lower_errors,
        upper_errors,
    ]),
    capsize=4,
)

plt.xlabel("Top-1 contradiction accuracy")
plt.ylabel("Method")
plt.title(
    "SNLI contradiction retrieval accuracy"
)
plt.xlim(0, 1)

plt.tight_layout()

ACCURACY_FIGURE_PATH = (
    FIGURES_DIR /
    "snli_top1_contradiction_accuracy.png"
)

plt.savefig(
    ACCURACY_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", ACCURACY_FIGURE_PATH)

In [ ]:
mrr_plot = summary_df.sort_values(
    "mean_reciprocal_rank",
    ascending=True,
)

plt.figure(figsize=(11, 7))

plt.barh(
    mrr_plot["method"],
    mrr_plot["mean_reciprocal_rank"],
)

plt.xlabel("Mean Reciprocal Rank")
plt.ylabel("Method")
plt.title(
    "SNLI contradiction retrieval: MRR"
)
plt.xlim(0, 1)

plt.tight_layout()

MRR_FIGURE_PATH = (
    FIGURES_DIR /
    "snli_mean_reciprocal_rank.png"
)

plt.savefig(
    MRR_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", MRR_FIGURE_PATH)

In [ ]:
runtime_plot = summary_df.sort_values(
    "mean_runtime_seconds",
    ascending=True,
)

plt.figure(figsize=(11, 7))

plt.barh(
    runtime_plot["method"],
    runtime_plot["mean_runtime_seconds"],
)

plt.xlabel("Mean runtime per premise (seconds)")
plt.ylabel("Method")
plt.title("SNLI runtime comparison")

plt.tight_layout()

RUNTIME_FIGURE_PATH = (
    FIGURES_DIR /
    "snli_runtime_comparison.png"
)

plt.savefig(
    RUNTIME_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", RUNTIME_FIGURE_PATH)

In [ ]:
ours_examples = results_df[
    results_df["method"].eq("ours_nli_gpt")
].copy()

success_examples = ours_examples[
    ours_examples["top1_contradiction_correct"].eq(1)
].head(10)

failure_examples = ours_examples[
    ours_examples["top1_contradiction_correct"].eq(0)
].head(10)

print("SUCCESS EXAMPLES")
display(
    success_examples[
        [
            "case_id",
            "premise",
            "selected_hypothesis",
            "selected_gold_label_name",
            "method_details",
        ]
    ]
)

print("FAILURE EXAMPLES")
display(
    failure_examples[
        [
            "case_id",
            "premise",
            "selected_hypothesis",
            "selected_gold_label_name",
            "method_details",
        ]
    ]
)